# DigitLens — Notebook Machine Learning
**Mata Kuliah Pembelajaran Mesin · Project Akhir (ML2026)**

Notebook ini mencakup seluruh tahap ML pada project: *loading dataset → EDA → preprocessing → pelatihan 3 algoritma → evaluasi → perbandingan → analisis → pemilihan model → pengujian end-to-end*.

| | |
|---|---|
| **Masalah** | Klasifikasi citra digit tulisan/cetak (0–9) sebagai inti dari pengenalan angka multi-digit |
| **Jenis tugas** | Supervised learning – *multi-class classification* (10 kelas) |
| **Dataset** | MNIST (LeCun et al.), tersedia lewat `keras.datasets.mnist` |
| **Algoritma dibandingkan** | CNN, SVM (RBF, dengan PCA), Random Forest |
| **Metrik** | Accuracy, Precision, Recall, F1-Score (macro), Confusion Matrix, waktu latih/inferensi |

> **Catatan untuk kelompok:** sel bertanda **✍️ ISI** adalah interpretasi yang harus Anda tulis sendiri setelah melihat hasil
> yang keluar dari notebook ini. Jangan menyalin angka dari tempat lain; semua hasil harus berasal dari eksekusi notebook.

## 0. Setup & konfigurasi

In [ ]:
import os, time, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

import keras
from keras import layers
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

# Mode cepat hanya untuk uji coba kode (set variabel lingkungan DIGITLENS_QUICK=1).
# Untuk hasil laporan, jalankan dalam mode normal (QUICK = False).
QUICK = os.environ.get("DIGITLENS_QUICK") == "1"
CNN_EPOCHS = 2 if QUICK else 15
RF_TREES = 40 if QUICK else 200
TRAIN_LIMIT = 6000 if QUICK else None      # batasi jumlah data latih (hanya mode cepat)
SVM_MAX_TRAIN = 3000 if QUICK else None    # None = pakai seluruh data latih

# Lokasi proyek (notebook boleh dijalankan dari folder notebooks/ atau root proyek)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPORT_DIR = ROOT / "reports"
FIG_DIR = REPORT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
# Model hasil latih disimpan dengan nama BARU agar model_cnn.keras yang sedang dipakai aplikasi tidak tertimpa.
SAVE_CNN_PATH = ROOT / "models" / "model_cnn_retrained.keras"

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

print("Keras", keras.__version__, "| backend:", keras.backend.backend(), "| QUICK =", QUICK)
print("ROOT:", ROOT)

## 1. Dataset
**Sumber:** MNIST — LeCun, Y., Cortes, C., & Burges, C. (1998), *The MNIST database of handwritten digits*. Dimuat melalui `keras.datasets.mnist`.

Setiap sampel adalah citra grayscale **28×28 piksel** (784 fitur, nilai 0–255) dengan label digit **0–9** (target).
Dataset sudah memiliki pembagian resmi: 60.000 data latih dan 10.000 data uji.

In [ ]:
from keras.datasets import mnist
(X_train_full, y_train_full), (X_test, y_test) = mnist.load_data()

if TRAIN_LIMIT:  # hanya mode cepat
    X_train_full, y_train_full = X_train_full[:TRAIN_LIMIT], y_train_full[:TRAIN_LIMIT]

info = pd.DataFrame({
    "Bagian": ["Data latih (penuh)", "Data uji"],
    "Jumlah citra": [len(X_train_full), len(X_test)],
    "Ukuran citra": [X_train_full.shape[1:], X_test.shape[1:]],
    "Jumlah fitur (piksel)": [int(np.prod(X_train_full.shape[1:]))] * 2,
    "Tipe data": [str(X_train_full.dtype), str(X_test.dtype)],
    "Rentang nilai": [f"{X_train_full.min()}–{X_train_full.max()}", f"{X_test.min()}–{X_test.max()}"],
})
info

## 2. Exploratory Data Analysis (EDA)

### 2.1 Kualitas data: missing value, duplikat, dan kebocoran train–test

In [ ]:
n_missing = int(np.isnan(X_train_full.astype("float32")).sum() + np.isnan(X_test.astype("float32")).sum())
train_hashes = [x.tobytes() for x in X_train_full]
test_hashes = [x.tobytes() for x in X_test]
dup_in_train = len(train_hashes) - len(set(train_hashes))
overlap = len(set(train_hashes) & set(test_hashes))

print(f"Missing value                : {n_missing}")
print(f"Citra duplikat di data latih : {dup_in_train}")
print(f"Citra identik latih vs uji   : {overlap}  (harus 0 agar tidak ada kebocoran)")

**✍️ ISI – Interpretasi:** jelaskan apakah ada missing value/duplikat, dan mengapa pengecekan tumpang tindih data latih–uji penting.

### 2.2 Distribusi kelas (target)

In [ ]:
counts = pd.Series(y_train_full).value_counts().sort_index()
ratio = counts.max() / counts.min()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
sns.barplot(x=counts.index, y=counts.values, ax=ax[0], color="#2563eb")
ax[0].set(title="Distribusi kelas – data latih", xlabel="Digit", ylabel="Jumlah citra")
test_counts = pd.Series(y_test).value_counts().sort_index()
sns.barplot(x=test_counts.index, y=test_counts.values, ax=ax[1], color="#4f46e5")
ax[1].set(title="Distribusi kelas – data uji", xlabel="Digit", ylabel="Jumlah citra")
savefig("eda_class_distribution")
print(f"Kelas terbanyak: {counts.idxmax()} ({counts.max()}) · tersedikit: {counts.idxmin()} ({counts.min()}) · rasio = {ratio:.2f}")

**Cara membaca:** tinggi batang = jumlah citra per digit. Rasio kelas terbanyak : tersedikit mendekati 1 berarti dataset **relatif seimbang**,
sehingga *accuracy* masih layak dipakai, tetapi tetap lengkapi dengan precision/recall/F1 per kelas.

**✍️ ISI – Interpretasi:** apakah kelas tidak seimbang? Apakah perlu *class weight*/*resampling*? Berikan alasan berdasarkan rasio di atas.

### 2.3 Contoh citra dan rata-rata citra per kelas

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(14, 3.4))
for d in range(10):
    idx = np.where(y_train_full == d)[0][:2]
    for r, i in enumerate(idx):
        axes[r, d].imshow(X_train_full[i], cmap="gray"); axes[r, d].axis("off")
        if r == 0: axes[r, d].set_title(str(d))
fig.suptitle("Contoh citra tiap kelas", y=1.02)
savefig("eda_samples")

fig, axes = plt.subplots(1, 10, figsize=(14, 1.9))
for d in range(10):
    axes[d].imshow(X_train_full[y_train_full == d].mean(axis=0), cmap="magma"); axes[d].axis("off"); axes[d].set_title(str(d))
fig.suptitle("Rata-rata citra per kelas", y=1.08)
savefig("eda_mean_images")

**Cara membaca:** citra rata-rata menunjukkan "bentuk umum" tiap digit; area terang = piksel yang sering terisi.
Digit yang rata-ratanya mirip (misalnya 4 dan 9, atau 3 dan 5) berpotensi sering tertukar oleh model.

**✍️ ISI – Interpretasi:** sebutkan pasangan digit yang menurut Anda mirip dan alasannya. Bandingkan nanti dengan *confusion matrix* (bagian 6).

### 2.4 Distribusi intensitas piksel dan jumlah 'tinta' per kelas

In [ ]:
flat_all = X_train_full.reshape(len(X_train_full), -1)
ink = flat_all.sum(axis=1) / 255.0                      # jumlah tinta ≈ ketebalan goresan
nonzero = flat_all[flat_all > 0]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].hist(nonzero, bins=50, color="#2563eb")
ax[0].set(title="Histogram intensitas piksel (piksel ≠ 0)", xlabel="Nilai piksel", ylabel="Frekuensi")
sns.boxplot(x=y_train_full, y=ink, ax=ax[1], color="#93c5fd", fliersize=1.5)
ax[1].set(title="Jumlah tinta per kelas (boxplot)", xlabel="Digit", ylabel="Σ piksel / 255")
savefig("eda_pixel_ink")

print(f"Piksel bernilai 0 (latar): {(flat_all == 0).mean():.1%} dari seluruh piksel")
print("Median tinta per kelas:", dict(pd.Series(ink).groupby(y_train_full).median().round(1)))

**Cara membaca:** histogram memperlihatkan mayoritas piksel adalah latar (0) dan sebagian kecil bernilai tinggi (goresan).
Boxplot memperlihatkan variasi ketebalan goresan; kelas "1" biasanya paling sedikit tintanya, kelas "0"/"8" paling banyak.
Titik di luar whisker adalah *outlier* (goresan sangat tebal/tipis) — tidak dibuang karena itu variasi tulisan yang sah.

**✍️ ISI – Interpretasi:** kaitkan perbedaan tinta antar kelas dengan kemungkinan kesulitan model pada gambar nyata (misalnya goresan tipis pada hasil crop).

## 3. Preprocessing
1. **Normalisasi** piksel ke rentang 0–1 (`/255`) — jaringan saraf dan SVM sensitif terhadap skala fitur. Ini **sama** dengan `PIXEL_SCALE` pada `digitlens/predictor.py`.
2. **Pembagian data:** data uji resmi MNIST (10.000) **tidak disentuh** sampai evaluasi akhir. Data latih dipecah **90% train : 10% validasi** secara *stratified* (proporsi kelas terjaga). Validasi dipakai untuk *early stopping* dan memilih hiperparameter.
3. **Bentuk data:** CNN memakai `(28, 28, 1)`; SVM/Random Forest memakai vektor 784 fitur (flatten).
4. **PCA (50 komponen) khusus SVM** untuk mempercepat pelatihan; PCA di-*fit* **hanya pada data latih** agar tidak ada kebocoran data.
5. Tidak ada *encoding* kategorikal, penanganan missing value, atau penghapusan outlier karena data bersih dan seimbang (lihat EDA).

In [ ]:
X_tr_raw, X_val_raw, y_tr, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.10, stratify=y_train_full, random_state=SEED)

def norm(X):  # 0..255 -> 0..1
    return X.astype("float32") / 255.0

X_tr, X_val, X_te = norm(X_tr_raw), norm(X_val_raw), norm(X_test)
flat = lambda X: X.reshape(len(X), -1)

print(f"Train: {X_tr.shape}  Validasi: {X_val.shape}  Test: {X_te.shape}")
print("Proporsi kelas train vs val (selisih maks):",
      float(np.abs(pd.Series(y_tr).value_counts(normalize=True).sort_index()
                   - pd.Series(y_val).value_counts(normalize=True).sort_index()).max().round(4)))

# PCA hanya di-fit pada data latih
pca = PCA(n_components=50, random_state=SEED).fit(flat(X_tr))
print(f"PCA 50 komponen menjelaskan {pca.explained_variance_ratio_.sum():.1%} varians data latih")

## 4. Pengembangan model
**Alasan pemilihan algoritma** (dibandingkan pada data yang sama):

| Algoritma | Alasan dipilih | Perkiraan kelemahan |
|---|---|---|
| **CNN** | Mempelajari pola spasial lokal (goresan, lengkungan) langsung dari piksel; standar untuk citra | Butuh waktu latih lebih besar, kurang mudah diinterpretasi |
| **SVM (RBF)** | Kuat pada data berdimensi tinggi dengan batas kelas non-linear; baseline klasik kuat untuk MNIST | Lambat pada data besar, tidak memanfaatkan struktur spasial |
| **Random Forest** | Baseline *ensemble* yang cepat dilatih, tahan overfitting, dan memberi *feature importance* | Memperlakukan piksel sebagai fitur independen |

Seluruh model memakai **data latih/validasi yang sama** dan dievaluasi pada **data uji yang sama**.

### 4.1 CNN

In [ ]:
def build_cnn():
    # Arsitektur SAMA dengan models/model_cnn.keras yang dipakai aplikasi
    return keras.Sequential([
        layers.Input((28, 28, 1)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax"),
    ], name="cnn_digit")

cnn = build_cnn()
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.summary()

t0 = time.perf_counter()
hist = cnn.fit(
    X_tr[..., None], y_tr, validation_data=(X_val[..., None], y_val),
    epochs=CNN_EPOCHS, batch_size=128, verbose=2,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)])
cnn_fit_time = time.perf_counter() - t0
print(f"Waktu latih CNN: {cnn_fit_time:.1f} detik")

### 4.2 SVM (RBF) + PCA
Parameter `C` dipilih memakai **data validasi** (bukan data uji) pada subset 10.000 data latih agar cepat, lalu model akhir dilatih pada seluruh data latih.

In [ ]:
Z_tr, Z_val, Z_te = pca.transform(flat(X_tr)), pca.transform(flat(X_val)), pca.transform(flat(X_te))

# Seleksi C berdasarkan akurasi validasi (subset agar cepat)
sub = min(10000, len(Z_tr))
rows = []
for C in [1, 5, 10]:
    m = SVC(kernel="rbf", C=C, gamma="scale").fit(Z_tr[:sub], y_tr[:sub])
    rows.append({"C": C, "val_accuracy": accuracy_score(y_val, m.predict(Z_val))})
svm_tuning = pd.DataFrame(rows)
best_C = int(svm_tuning.sort_values("val_accuracy", ascending=False).iloc[0]["C"])
display(svm_tuning)
print("C terpilih:", best_C)

n_svm = SVM_MAX_TRAIN or len(Z_tr)
t0 = time.perf_counter()
svm = SVC(kernel="rbf", C=best_C, gamma="scale").fit(Z_tr[:n_svm], y_tr[:n_svm])
svm_fit_time = time.perf_counter() - t0
print(f"SVM dilatih pada {n_svm} data · waktu latih: {svm_fit_time:.1f} detik")

### 4.3 Random Forest

In [ ]:
t0 = time.perf_counter()
rf = RandomForestClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=SEED).fit(flat(X_tr), y_tr)
rf_fit_time = time.perf_counter() - t0
print(f"Random Forest ({RF_TREES} pohon) · waktu latih: {rf_fit_time:.1f} detik")

## 5. Evaluasi pada data uji
Fungsi prediksi disatukan agar ketiga model diuji dengan cara yang identik. Data uji **hanya dipakai di bagian ini dan seterusnya**.

In [ ]:
def predict_cnn(X): return np.argmax(cnn.predict(X[..., None], batch_size=512, verbose=0), axis=1)
def predict_svm(X): return svm.predict(pca.transform(flat(X)))
def predict_rf(X):  return rf.predict(flat(X))

MODELS = {"CNN": predict_cnn, "SVM (RBF+PCA)": predict_svm, "Random Forest": predict_rf}
FIT_TIME = {"CNN": cnn_fit_time, "SVM (RBF+PCA)": svm_fit_time, "Random Forest": rf_fit_time}

rows, preds = [], {}
for name, fn in MODELS.items():
    t0 = time.perf_counter(); y_pred = fn(X_te); infer = time.perf_counter() - t0
    preds[name] = y_pred
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (macro)": precision_score(y_test, y_pred, average="macro"),
        "Recall (macro)": recall_score(y_test, y_pred, average="macro"),
        "F1-Score (macro)": f1_score(y_test, y_pred, average="macro"),
        "Waktu latih (s)": FIT_TIME[name],
        "Inferensi / 1000 citra (s)": infer / len(X_te) * 1000,
    })
results = pd.DataFrame(rows).set_index("Model")
results.round(4)

### 5.1 Confusion matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, (name, y_pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax, annot_kws={"size": 7})
    ax.set(title=f"{name} · acc={accuracy_score(y_test, y_pred):.4f}", xlabel="Prediksi", ylabel="Sebenarnya")
savefig("confusion_matrices")

**Cara membaca:** diagonal = prediksi benar; sel di luar diagonal = kesalahan. Baris = label sebenarnya, kolom = prediksi.
**✍️ ISI – Interpretasi:** pasangan digit mana yang paling sering tertukar pada tiap model? Apakah polanya sama antar model?

### 5.2 Laporan per kelas

In [ ]:
for name, y_pred in preds.items():
    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, digits=4))

## 6. Perbandingan model

In [ ]:
display(results.round(4).sort_values("F1-Score (macro)", ascending=False))
results.round(6).to_csv(REPORT_DIR / "model_comparison.csv")

metric_cols = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1-Score (macro)"]
ax = results[metric_cols].plot(kind="bar", figsize=(9, 4), ylim=(max(0, results[metric_cols].min().min() - 0.02), 1.0), rot=0)
ax.set_title("Perbandingan metrik pada data uji"); ax.set_ylabel("Skor")
savefig("model_comparison")

## 7. Analisis model
### 7.1 Overfitting / underfitting
Dibandingkan akurasi pada data latih, validasi, dan uji. **Selisih latih–uji yang besar** mengindikasikan overfitting;
akurasi rendah di semua bagian mengindikasikan underfitting.

In [ ]:
rng = np.random.default_rng(SEED)
tr_idx = rng.choice(len(X_tr), size=min(10000, len(X_tr)), replace=False)
rows = []
for name, fn in MODELS.items():
    tr_acc = accuracy_score(y_tr[tr_idx], fn(X_tr[tr_idx]))
    val_acc = accuracy_score(y_val, fn(X_val))
    te_acc = results.loc[name, "Accuracy"]
    rows.append({"Model": name, "Akurasi train": tr_acc, "Akurasi validasi": val_acc,
                 "Akurasi test": te_acc, "Selisih train–test": tr_acc - te_acc})
overfit = pd.DataFrame(rows).set_index("Model")
display(overfit.round(4))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="validasi")
ax[0].set(title="CNN – loss", xlabel="Epoch"); ax[0].legend()
ax[1].plot(hist.history["accuracy"], label="train"); ax[1].plot(hist.history["val_accuracy"], label="validasi")
ax[1].set(title="CNN – accuracy", xlabel="Epoch"); ax[1].legend()
savefig("cnn_learning_curve")

**✍️ ISI – Interpretasi:** model mana yang selisih train–test-nya paling besar? Apakah kurva CNN menunjukkan overfitting (val loss naik sementara train loss turun)?

### 7.2 Model terbaik, terendah, dan kesalahan prediksi

In [ ]:
best_name = results["F1-Score (macro)"].idxmax()
worst_name = results["F1-Score (macro)"].idxmin()
print(f"Terbaik (F1 macro): {best_name} · Terendah: {worst_name}")

# Pasangan digit yang paling sering tertukar pada model terbaik
cm = confusion_matrix(y_test, preds[best_name]); np.fill_diagonal(cm, 0)
pairs = sorted([(cm[i, j], i, j) for i in range(10) for j in range(10) if cm[i, j] > 0], reverse=True)[:8]
print(f"\nPasangan paling sering tertukar ({best_name}):")
for n, i, j in pairs: print(f"  sebenarnya {i} → diprediksi {j}: {n} kali")

# Contoh citra yang salah diprediksi (dengan confidence jika CNN)
wrong = np.where(preds[best_name] != y_test)[0][:12]
proba = cnn.predict(X_te[wrong][..., None], verbose=0) if best_name == "CNN" else None
fig, axes = plt.subplots(2, 6, figsize=(12, 4.6))
for k, (ax, i) in enumerate(zip(axes.ravel(), wrong)):
    ax.imshow(X_te[i], cmap="gray"); ax.axis("off")
    conf = f" ({proba[k].max():.0%})" if proba is not None else ""
    ax.set_title(f"asli {y_test[i]} → {preds[best_name][i]}{conf}", fontsize=9)
fig.suptitle(f"Contoh kesalahan prediksi – {best_name}")
savefig("misclassified_examples")

**✍️ ISI – Interpretasi:** lihat contoh kesalahan di atas. Apakah kesalahan wajar (tulisan ambigu) atau menunjukkan kelemahan model? Sebutkan alasannya.

### 7.3 Feature importance (Random Forest)

In [ ]:
imp = rf.feature_importances_.reshape(28, 28)
plt.figure(figsize=(4.8, 4.2))
plt.imshow(imp, cmap="hot"); plt.colorbar(label="Importance"); plt.axis("off")
plt.title("Importance tiap piksel – Random Forest")
savefig("rf_feature_importance")

**Cara membaca:** area terang = piksel yang paling banyak dipakai pohon untuk memisahkan kelas (umumnya bagian tengah citra tempat goresan berada;
piksel tepi hampir selalu kosong sehingga tidak informatif).
**✍️ ISI – Interpretasi:** jelaskan hubungan pola importance dengan bentuk digit.

### 7.4 Uji ketahanan (robustness)
Aplikasi menerima gambar dari dunia nyata, bukan hanya citra MNIST yang rapi. Di sini data uji diberi gangguan
(rotasi, pergeseran, *noise*, goresan lebih tebal/tipis) untuk melihat model mana yang paling tahan. Dipakai subset 2.000 citra uji agar cepat.

In [ ]:
sub_idx = np.random.default_rng(SEED).choice(len(X_te), size=min(2000, len(X_te)), replace=False)
Xs, ys = X_te[sub_idx], y_test[sub_idx]

def rotate(img, deg):
    M = cv2.getRotationMatrix2D((13.5, 13.5), deg, 1.0); return cv2.warpAffine(img, M, (28, 28))
def shift(img, dx, dy):
    return cv2.warpAffine(img, np.float32([[1, 0, dx], [0, 1, dy]]), (28, 28))
def noise(img, s=0.2):
    return np.clip(img + np.random.default_rng(0).normal(0, s, img.shape), 0, 1).astype("float32")
def thicken(img): return cv2.dilate(img, np.ones((2, 2), np.uint8))
def thin(img):    return cv2.erode(img, np.ones((2, 2), np.uint8))

PERTURB = {
    "Asli": lambda a: a,
    "Rotasi 15°": lambda a: rotate(a, 15),
    "Geser 3 px": lambda a: shift(a, 3, 3),
    "Noise σ=0.2": noise,
    "Goresan tebal": thicken,
    "Goresan tipis": thin,
}
rows = []
for pname, pf in PERTURB.items():
    Xp = np.stack([pf(a) for a in Xs]).astype("float32")
    row = {"Gangguan": pname}
    for name, fn in MODELS.items():
        row[name] = accuracy_score(ys, fn(Xp))
    rows.append(row)
robust = pd.DataFrame(rows).set_index("Gangguan")
display(robust.round(4))
robust.round(6).to_csv(REPORT_DIR / "robustness.csv")
ax = robust.plot(kind="bar", figsize=(10, 4), rot=15, ylim=(0, 1.0)); ax.set_ylabel("Accuracy")
ax.set_title("Ketahanan model terhadap gangguan")
savefig("robustness")

**✍️ ISI – Interpretasi:** gangguan apa yang paling menurunkan akurasi dan pada model mana? Apa implikasinya untuk aplikasi yang menerima foto/tulisan nyata?

## 8. Pemilihan model untuk aplikasi
Model **tidak dipilih hanya karena accuracy tertinggi**. Pertimbangkan: F1-macro, selisih train–test (overfitting), ketahanan terhadap gangguan,
waktu inferensi, dan ukuran/kemudahan integrasi dengan aplikasi (Keras `.keras` + preprocessing `predictor.py`).

In [ ]:
summary = results[["Accuracy", "F1-Score (macro)", "Inferensi / 1000 citra (s)"]].join(overfit[["Selisih train–test"]]).join(robust.drop(index="Asli").mean().rename("Rata-rata akurasi saat diganggu"))
display(summary.round(4))
print("Model dengan F1 macro tertinggi:", results["F1-Score (macro)"].idxmax())

**✍️ ISI – Alasan pemilihan model:** tuliskan model yang dipilih untuk aplikasi dan alasannya berdasarkan tabel di atas (bukan hanya accuracy).

In [ ]:
# Simpan CNN yang baru dilatih. Ganti nama/pindahkan ke models/model_cnn.keras HANYA jika Anda ingin
# aplikasi memakai model ini dan sudah menguji pipeline end-to-end (bagian 9).
SAVE_CNN_PATH.parent.mkdir(parents=True, exist_ok=True)
cnn.save(SAVE_CNN_PATH)
(REPORT_DIR / "metrics_summary.json").write_text(json.dumps({
    "results": results.round(6).to_dict(), "svm_C": best_C, "cnn_epochs_run": len(hist.history["loss"]), "quick_mode": QUICK,
}, indent=2, ensure_ascii=False), encoding="utf-8")
print("Model disimpan ke:", SAVE_CNN_PATH)

## 9. Pengujian end-to-end (gambar nyata → angka)
Akurasi pada data uji MNIST **tidak menjamin** akurasi pada gambar hasil segmentasi. Bagian ini menguji pipeline aplikasi yang sebenarnya
(`segmentation.py` + `predictor.py` + `models/model_cnn.keras`) pada folder `examples/testset/`.

Format nama file: `<angka_sebenarnya>__<keterangan>.png` (contoh `90417__foto1.png`).
`python scripts/make_synthetic_testset.py` membuat 40 gambar sintetis sebagai pelengkap, **tetapi untuk laporan tambahkan juga gambar
nyata milik kelompok** (foto/tulisan tangan/screenshot) agar hasilnya mewakili penggunaan sebenarnya.

In [ ]:
import sys
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "scripts"))
TESTSET = ROOT / "examples" / "testset"

if not TESTSET.exists() or not any(TESTSET.glob("*__*.*")):
    print("Folder examples/testset/ belum berisi gambar. Jalankan scripts/make_synthetic_testset.py lalu ulangi sel ini.")
else:
    import eval_end_to_end as e2e
    e2e_df = e2e.evaluate_testset(TESTSET)
    for k, v in e2e.summarize(e2e_df).items(): print(f"{k}: {v}")
    e2e_df.to_csv(REPORT_DIR / "end_to_end_results.csv", index=False)
    print("\nKasus gagal:")
    display(e2e_df[~e2e_df["exact_match"]][["file", "truth", "predicted", "n_candidates", "min_confidence"]].round(3))

**✍️ ISI – Analisis end-to-end:** bedakan kegagalan karena **segmentasi** (jumlah kandidat ≠ jumlah digit, misalnya digit menempel) dan karena
**klasifikasi** (jumlah benar tetapi digit salah). Apakah confidence rendah (<0.8) memang berkorelasi dengan kesalahan?

## 10. Keterbatasan dan pengembangan lanjutan
**✍️ ISI** berdasarkan hasil Anda sendiri, misalnya:
- **Dataset:** MNIST berisi tulisan tangan di latar bersih; domain gambar nyata (font cetak, foto, noise) berbeda → lihat hasil bagian 7.4 dan 9.
- **Model:** apakah ada pasangan digit yang konsisten tertukar? Apakah augmentasi data (rotasi, variasi ketebalan, font cetak) dapat membantu?
- **Aplikasi:** segmentasi *connected-component* gagal untuk digit menempel/multi-baris.
- **Pengembangan:** augmentasi data, fine-tuning pada dataset font/foto sendiri, segmentasi berbasis *projection profile*/watershed, *confidence calibration*.